# License Plate Detection — Model 1: YOLOv8s
**CMPS 261 — Machine Learning Project**

YOLOv8s is a single-stage detector that predicts bounding boxes and class probabilities in one forward pass — very fast and accurate.

This notebook runs **both locally and on Google Colab**. On Colab, select Runtime → Change runtime type → T4 GPU for fast training.

## 1. Environment Setup

Auto-detects Colab vs local and configures paths accordingly.

In [ ]:
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'Running on: {"Google Colab" if IN_COLAB else "Local"}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile
    zip_path = '/content/drive/MyDrive/license_plate_data.zip'
    if not os.path.exists('/content/data/archive'):
        if not os.path.exists(zip_path):
            raise RuntimeError('license_plate_data.zip not found in Google Drive root.')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/content/')
        print('Extracted dataset archive.')
    BASE_DIR = '/content'
else:
    BASE_DIR = '..'

def prepare_purified_dataset(base_dir, seed=42):
    """Create a deduplicated YOLO split from data/archive."""
    import hashlib
    import random
    import shutil
    import xml.etree.ElementTree as ET
    from pathlib import Path

    base_dir = Path(base_dir)
    img_dir = base_dir / 'data' / 'archive' / 'images'
    ann_dir = base_dir / 'data' / 'archive' / 'annotations'
    yolo_dir = base_dir / 'data' / 'yolo'

    if not img_dir.exists() or not ann_dir.exists():
        raise RuntimeError(f'Raw dataset not found under {base_dir / "data" / "archive"}')

    def file_md5(path):
        h = hashlib.md5()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 20), b''):
                h.update(chunk)
        return h.hexdigest()

    def parse_xml(xml_path):
        root = ET.parse(xml_path).getroot()
        filename = root.find('filename').text
        img_w = int(root.find('size/width').text)
        img_h = int(root.find('size/height').text)
        boxes = []
        for obj in root.findall('object'):
            boxes.append((
                int(obj.find('bndbox/xmin').text),
                int(obj.find('bndbox/ymin').text),
                int(obj.find('bndbox/xmax').text),
                int(obj.find('bndbox/ymax').text),
            ))
        return filename, img_w, img_h, boxes

    def voc_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h):
        cx = (xmin + xmax) / 2 / img_w
        cy = (ymin + ymax) / 2 / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        return cx, cy, w, h

    xml_files = sorted(ann_dir.glob('*.xml'))
    hash_to_xmls = {}
    for xml_path in xml_files:
        filename, *_ = parse_xml(xml_path)
        img_path = img_dir / filename
        if img_path.exists():
            hash_to_xmls.setdefault(file_md5(img_path), []).append(xml_path)

    unique_xmls = sorted(min(group) for group in hash_to_xmls.values())
    print(f'Dedup: {len(xml_files)} XMLs -> {len(unique_xmls)} unique images ({len(xml_files) - len(unique_xmls)} duplicate copies removed)')

    random.seed(seed)
    random.shuffle(unique_xmls)
    n = len(unique_xmls)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    splits = {
        'train': unique_xmls[:n_train],
        'val': unique_xmls[n_train:n_train + n_val],
        'test': unique_xmls[n_train + n_val:],
    }

    if yolo_dir.exists():
        shutil.rmtree(yolo_dir)
    for split in splits:
        (yolo_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (yolo_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    for split, files in splits.items():
        for xml_path in files:
            filename, img_w, img_h, boxes = parse_xml(xml_path)
            shutil.copy2(img_dir / filename, yolo_dir / 'images' / split / filename)
            label_path = yolo_dir / 'labels' / split / f'{Path(filename).stem}.txt'
            with open(label_path, 'w') as f:
                for xmin, ymin, xmax, ymax in boxes:
                    cx, cy, w, h = voc_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h)
                    f.write(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')

    yaml_path = yolo_dir / 'dataset.yaml'
    yaml_root = str(yolo_dir.resolve()) if str(base_dir) != '/content' else '/content/data/yolo'
    with open(yaml_path, 'w') as f:
        f.write(f'path: {yaml_root}\n')
        f.write('train: images/train\n')
        f.write('val:   images/val\n')
        f.write('test:  images/test\n\n')
        f.write('nc: 1\n')
        f.write("names: ['licence']\n")

    seen = {}
    for split in splits:
        for img in (yolo_dir / 'images' / split).iterdir():
            h = file_md5(img)
            if h in seen and seen[h] != split:
                raise RuntimeError(f'Cross-split duplicate after dedup: {img.name} in {split} matches {seen[h]}')
            seen[h] = split

    print('Data prepared:')
    for split, files in splits.items():
        print(f'  {split:<5}: {len(files)} images')
    print(f'  YAML  : {yaml_path}')
    print('  Cross-split duplicates: 0 (verified)')
    return str(yaml_path)

YAML_PATH = prepare_purified_dataset(BASE_DIR)

YOLO_IMGSZ = 960  # Higher resolution helps small plate localization; use 640 for faster baseline reruns.

if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', 'ultralytics', '-q'], check=True)
    from google.colab import files
    MODEL_DIR = '/content/models'
    RESULTS_DIR = '/content/results'
    WEIGHTS_OUT = f'/content/models/yolov8s_imgsz{YOLO_IMGSZ}/weights/best.pt'
    WEIGHTS_SAVE = '/content/yolov8s_best.pt'
    TEST_IMG_DIR = '/content/data/yolo/images/test'
    DEVICE = 0
else:
    import torch
    MODEL_DIR = '../models'
    RESULTS_DIR = '../results'
    WEIGHTS_OUT = f'../models/yolov8s_imgsz{YOLO_IMGSZ}/weights/best.pt'
    WEIGHTS_SAVE = '../models/yolov8s_best.pt'
    TEST_IMG_DIR = '../data/yolo/images/test'
    DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
yaml_path = YAML_PATH
print(f'Dataset YAML : {yaml_path}')
print(f'Device       : {DEVICE}')


## 2. Train YOLOv8s

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data     = yaml_path,
    epochs   = 100,
    imgsz    = YOLO_IMGSZ,
    batch    = 16,
    device   = DEVICE,
    project  = MODEL_DIR,
    name     = f'yolov8s_imgsz{YOLO_IMGSZ}',
    exist_ok = True,
    verbose  = True,
)

## 3. Evaluate on Test Set

In [ ]:
test_metrics = model.val(split='test', imgsz=YOLO_IMGSZ)
yolo_f1 = 2 * test_metrics.box.mp * test_metrics.box.mr / (test_metrics.box.mp + test_metrics.box.mr + 1e-6)
print(f'Test mAP@0.5      : {test_metrics.box.map50:.4f}')
print(f'Test mAP@0.5:0.95 : {test_metrics.box.map:.4f}')
print(f'Test Precision    : {test_metrics.box.mp:.4f}')
print(f'Test Recall       : {test_metrics.box.mr:.4f}')
print(f'Test F1           : {yolo_f1:.4f}')

## 4. Save Best Weights

In [ ]:
import os
import shutil

if not os.path.exists(WEIGHTS_OUT):
    raise RuntimeError(f'Trained YOLO weight not found: {WEIGHTS_OUT}. Run the training cell first.')
shutil.copy(WEIGHTS_OUT, WEIGHTS_SAVE)
print(f'Saved weights to: {WEIGHTS_SAVE}')


## 5. Visualise Predictions on Test Images

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

if not os.path.exists(WEIGHTS_SAVE):
    raise RuntimeError(f'Best YOLO weight not found: {WEIGHTS_SAVE}. Run the training/save cells first.')

test_pool = os.listdir(TEST_IMG_DIR)
test_images = random.sample(test_pool, min(8, len(test_pool)))
model_best = YOLO(WEIGHTS_SAVE)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, fname in zip(axes, test_images):
    img_path = os.path.join(TEST_IMG_DIR, fname)
    result = model_best.predict(img_path, conf=0.25, verbose=False)[0]
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = box.conf[0].item()
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor='lime', facecolor='none'))
        ax.text(x1, y1-4, f'{conf:.2f}', color='lime', fontsize=8,
                bbox=dict(facecolor='black', alpha=0.4, pad=1))
    ax.set_title(fname, fontsize=7); ax.axis('off')

for ax in axes[len(test_images):]:
    ax.axis('off')

plt.suptitle('YOLOv8s - Predictions on Test Set', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'yolo_predictions.png'), dpi=150)
plt.show()
print(f'Saved: {RESULTS_DIR}/yolo_predictions.png')


## 6. Save Metrics

In [ ]:
import json

yolo_metrics = {
    'model'    : 'YOLOv8s',
    'epochs'   : 100,
    'imgsz'    : YOLO_IMGSZ,
    'map50'    : round(test_metrics.box.map50, 4),
    'map50_95' : round(test_metrics.box.map,   4),
    'precision': round(test_metrics.box.mp,    4),
    'recall'   : round(test_metrics.box.mr,    4),
    'f1'       : round(float(yolo_f1), 4),
}
metrics_path = os.path.join(RESULTS_DIR, 'yolo_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(yolo_metrics, f, indent=2)
print(json.dumps(yolo_metrics, indent=2))

## 7. Download Weights (Colab only)

In [ ]:
import os

if IN_COLAB:
    from google.colab import files
    if os.path.exists(WEIGHTS_SAVE):
        files.download(WEIGHTS_SAVE)
    else:
        print(f'Weight file not found yet: {WEIGHTS_SAVE}')
    if 'metrics_path' in globals() and os.path.exists(metrics_path):
        files.download(metrics_path)
    else:
        print('Metrics file not found yet.')
    print('Download step finished.')
else:
    print(f'Weights saved at: {WEIGHTS_SAVE}')
